# LeetCode 1305 — All Elements in Two Binary Search Trees bằng Python

Link đề: https://leetcode.com/problems/all-elements-in-two-binary-search-trees/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Search Tree, Sorting, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def getAllElements(self, root1: TreeNode | None, root2: TreeNode | None) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Nắm tính chất cốt lõi của **cây nhị phân tìm kiếm (BST)**: duyệt **inorder** (trái – gốc – phải) cho dãy **tăng dần**.
2. Viết được duyệt inorder **đệ quy** và **lặp bằng stack**, kể cả dạng **generator** (`yield`).
3. Thành thạo kỹ thuật **trộn hai mảng đã sắp xếp bằng hai con trỏ** (bước "merge" của merge sort) trong `O(m + n)`.
4. Kết hợp: **duyệt inorder song song hai cây** bằng hai stack, trộn "lười" mà không cần lưu hai danh sách trung gian.
5. Tránh lỗi: dùng sai thứ tự duyệt, quên phần còn dư khi trộn, xử lý cây rỗng, quên đẩy nhánh trái của cây con phải.

## 2. Đề bài & ví dụ

Cho hai cây nhị phân tìm kiếm `root1` và `root2`. Trả về danh sách **tất cả** số nguyên của cả hai cây, **sắp xếp tăng dần** (giữ cả các giá trị trùng nhau).

**Ví dụ 1:** `root1 = [2,1,4], root2 = [1,0,3]` → `[0,1,1,2,3,4]`

```
  cây 1:   2        cây 2:   1
          / \               / \
         1   4             0   3
```

**Ví dụ 2:** `root1 = [1,null,8], root2 = [8,1]` → `[1,1,8,8]`

**Ràng buộc:** số nút mỗi cây trong `[0, 5000]` (cây có thể **rỗng**), `-10^5 <= Node.val <= 10^5`.

In [1]:
import heapq
import random
import sys
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = trống)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


examples = [
    (([2, 1, 4], [1, 0, 3]), [0, 1, 1, 2, 3, 4]),
    (([1, None, 8], [8, 1]), [1, 1, 8, 8]),
]
for (v1, v2), want in examples:
    print(f"root1={v1}, root2={v2} -> mong đợi {want}")

root1=[2, 1, 4], root2=[1, 0, 3] -> mong đợi [0, 1, 1, 2, 3, 4]
root1=[1, None, 8], root2=[8, 1] -> mong đợi [1, 1, 8, 8]


## 3. Kiến thức nền

### 3.1 Cây nhị phân tìm kiếm (BST)

BST: với mọi nút, mọi giá trị ở **cây con trái** `<` giá trị nút `<` mọi giá trị ở **cây con phải** (đề LeetCode thường cho giá trị phân biệt trong một cây). Hệ quả quan trọng: **duyệt inorder ra dãy tăng dần**.

Hàm `insert_bst` dựng BST bằng cách chèn lần lượt — dùng để sinh test ngẫu nhiên.

In [2]:
def insert_bst(root, x):
    if root is None:
        return TreeNode(x)
    cur = root
    while True:
        if x < cur.val:
            if cur.left is None:
                cur.left = TreeNode(x); break
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = TreeNode(x); break
            cur = cur.right
    return root

def bst_from(values):
    root = None
    for x in values:
        root = insert_bst(root, x)
    return root

def show(root, indent=0):
    if root is None:
        return
    show(root.right, indent + 4)
    print(" " * indent + str(root.val))
    show(root.left, indent + 4)

show(bst_from([5, 3, 8, 1, 4, 9]))

        9
    8
5
        4
    3
        1


### 3.2 Ba kiểu duyệt DFS

- **Preorder:** gốc – trái – phải.
- **Inorder:** trái – gốc – phải ⇒ BST cho dãy tăng.
- **Postorder:** trái – phải – gốc.

In [3]:
def preorder(node, out):
    if node:
        out.append(node.val); preorder(node.left, out); preorder(node.right, out)
    return out

def inorder(node, out):
    if node:
        inorder(node.left, out); out.append(node.val); inorder(node.right, out)
    return out

def postorder(node, out):
    if node:
        postorder(node.left, out); postorder(node.right, out); out.append(node.val)
    return out

t = bst_from([5, 3, 8, 1, 4, 9])
print("preorder :", preorder(t, []))
print("inorder  :", inorder(t, []), "<- tăng dần")
print("postorder:", postorder(t, []))

preorder : [5, 3, 1, 4, 8, 9]
inorder  : [1, 3, 4, 5, 8, 9] <- tăng dần
postorder: [1, 4, 3, 9, 8, 5]


### 3.3 Inorder lặp bằng stack

Ý tưởng: "đi hết về bên trái, đẩy từng nút vào stack; lấy nút trên đỉnh ra (đó là nút nhỏ nhất chưa thăm), ghi nhận, rồi chuyển sang cây con phải và lặp lại". Không đệ quy ⇒ an toàn với cây lệch 5000 nút.

In [4]:
def inorder_iter(root):
    out, stack, cur = [], [], root
    while cur or stack:
        while cur:                 # đi hết về trái
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()          # nút nhỏ nhất chưa thăm
        out.append(cur.val)
        cur = cur.right            # sang cây con phải
    return out

print(inorder_iter(t))
print(inorder_iter(None), "<- cây rỗng")

[1, 3, 4, 5, 8, 9]
[] <- cây rỗng


### 3.4 Generator với `yield`

Hàm có `yield` trả về một **iterator lười**: mỗi lần `next()` chạy tiếp tới `yield` kế tiếp. Rất hợp với inorder: lấy dần từng phần tử nhỏ nhất mà không tạo cả danh sách.

In [5]:
def inorder_gen(root):
    stack, cur = [], root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        yield cur.val
        cur = cur.right

g = inorder_gen(t)
print(next(g), next(g), next(g), "| phần còn lại:", list(g))

1 3 4 | phần còn lại: [5, 8, 9]


### 3.5 Trộn hai mảng đã sắp xếp (two pointers) và `heapq.merge`

Hai con trỏ `i`, `j`: mỗi bước lấy phần tử nhỏ hơn trong `a[i]`, `b[j]`, tiến con trỏ đó. Khi một mảng hết, nối nốt phần còn lại của mảng kia. Thư viện chuẩn có sẵn `heapq.merge(*iterables)` làm việc này một cách lười.

In [6]:
def merge(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:])
    out.extend(b[j:])
    return out

print(merge([1, 2, 4], [0, 1, 3]))
print(list(heapq.merge([1, 2, 4], [0, 1, 3])))

[0, 1, 1, 2, 3, 4]
[0, 1, 1, 2, 3, 4]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**:
- Inorder cây 1 `[2,1,4]`: trái `1`, gốc `2`, phải `4` ⇒ `[1, 2, 4]`.
- Inorder cây 2 `[1,0,3]`: ⇒ `[0, 1, 3]`.
- Trộn hai dãy tăng:

| a (cây 1) | b (cây 2) | so sánh | lấy | kết quả |
|---|---|---|---|---|
| **1**,2,4 | **0**,1,3 | 1 vs 0 | 0 | [0] |
| **1**,2,4 | **1**,3 | 1 vs 1 | 1 (a) | [0,1] |
| **2**,4 | **1**,3 | 2 vs 1 | 1 | [0,1,1] |
| **2**,4 | **3** | 2 vs 3 | 2 | [0,1,1,2] |
| **4** | **3** | 4 vs 3 | 3 | [0,1,1,2,3] |
| **4** | – | b hết | 4 | [0,1,1,2,3,4] |

Kết quả khớp đề. Nhận xét: mỗi cây "tự nó" đã cho dãy tăng ⇒ không cần sort lại từ đầu.

In [7]:
for (v1, v2), want in examples:
    a, b = inorder_iter(build_tree(v1)), inorder_iter(build_tree(v2))
    print("inorder1 =", a, "| inorder2 =", b, "| trộn =", merge(a, b), "| mong đợi", want)

inorder1 = [1, 2, 4] | inorder2 = [0, 1, 3] | trộn = [0, 1, 1, 2, 3, 4] | mong đợi [0, 1, 1, 2, 3, 4]
inorder1 = [1, 8] | inorder2 = [1, 8] | trộn = [1, 1, 8, 8] | mong đợi [1, 1, 8, 8]


## 5. Cách 1 — Brute Force / baseline: gom hết rồi sort

**Ý tưởng:** không cần biết gì về BST — duyệt cả hai cây bằng bất kỳ cách nào (ở đây BFS), gom mọi giá trị vào một list rồi `sorted`. Đây chính là gợi ý của đề và **qua được** LeetCode, nhưng bỏ phí tính chất BST.

In [8]:
def all_elements_sort(root1, root2):
    vals = []
    for root in (root1, root2):
        q = deque([root] if root else [])
        while q:
            node = q.popleft()
            vals.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return sorted(vals)

for (v1, v2), want in examples:
    print(all_elements_sort(build_tree(v1), build_tree(v2)), "| mong đợi", want)

[0, 1, 1, 2, 3, 4] | mong đợi [0, 1, 1, 2, 3, 4]
[1, 1, 8, 8] | mong đợi [1, 1, 8, 8]


**Độ phức tạp:** gọi `m`, `n` là số nút hai cây, `N = m + n`.
- Time `O(N log N)` vì phải sort.
- Space `O(N)` cho danh sách kết quả (+ hàng đợi BFS).

**Nút thắt:** sort `O(N log N)` là thừa — mỗi cây đã "chứa sẵn" thứ tự tăng qua inorder. Thực tế Python dùng Timsort, vốn phát hiện các đoạn đã tăng sẵn; nếu ta nối hai dãy inorder rồi sort, Timsort chỉ còn `O(N)` — một mẹo thực dụng nhưng dựa vào chi tiết cài đặt.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** inorder một BST cho dãy **tăng dần** ⇒ hai cây cho hai dãy tăng `A`, `B`.

**Insight 2:** trộn hai dãy tăng chỉ tốn `O(|A| + |B|)` bằng hai con trỏ. **Quy tắc:** *nếu* `A[i] <= B[j]` *thì* lấy `A[i]`, tăng `i`; *ngược lại* lấy `B[j]`, tăng `j`. Khi một dãy hết, chép phần còn lại của dãy kia.

**Insight 3 (không cần dãy trung gian):** inorder lặp bằng stack cho phép lấy "phần tử nhỏ nhất kế tiếp" của mỗi cây **theo yêu cầu**. Phần tử nhỏ nhất chưa thăm của một cây luôn nằm ở **đỉnh stack** của cây đó. ⇒ So hai đỉnh stack, lấy cái nhỏ hơn, rồi "tiến" cây đó (đẩy nhánh trái của cây con phải).

**Invariant khi trộn:** `out` luôn là dãy tăng gồm đúng các phần tử nhỏ nhất đã lấy; mọi phần tử còn lại (ở cả hai phía) đều `>=` phần tử cuối của `out`.

In [9]:
# Minh hoạ Insight 3: đỉnh stack luôn là phần tử nhỏ nhất chưa thăm
root = bst_from([5, 3, 8, 1, 4, 9])
stack = []
cur = root
while cur:
    stack.append(cur); cur = cur.left
print("stack ban đầu (đáy->đỉnh):", [n.val for n in stack], "-> đỉnh =", stack[-1].val, "= min")
node = stack.pop()
cur = node.right
while cur:
    stack.append(cur); cur = cur.left
print("sau khi lấy", node.val, "-> stack:", [n.val for n in stack], "-> đỉnh =", stack[-1].val)

stack ban đầu (đáy->đỉnh): [5, 3, 1] -> đỉnh = 1 = min
sau khi lấy 1 -> stack: [5, 3] -> đỉnh = 3


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Inorder hai cây rồi trộn hai con trỏ

1. `a = inorder(root1)`, `b = inorder(root2)` (dùng bản lặp để tránh tràn đệ quy).
2. `merge(a, b)` như mục 3.5.

In [10]:
def all_elements_merge(root1, root2):
    a, b = inorder_iter(root1), inorder_iter(root2)
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:])                 # phần còn dư
    out.extend(b[j:])
    return out

for (v1, v2), want in examples:
    print(all_elements_merge(build_tree(v1), build_tree(v2)), "| mong đợi", want)

[0, 1, 1, 2, 3, 4] | mong đợi [0, 1, 1, 2, 3, 4]
[1, 1, 8, 8] | mong đợi [1, 1, 8, 8]


Code nộp LeetCode (cách 2):

In [11]:
class SolutionMerge:
    def getAllElements(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> List[int]:
        def inorder(root):
            out, stack, cur = [], [], root
            while cur or stack:
                while cur:
                    stack.append(cur)
                    cur = cur.left
                cur = stack.pop()
                out.append(cur.val)
                cur = cur.right
            return out
        a, b = inorder(root1), inorder(root2)
        i = j = 0
        res = []
        while i < len(a) and j < len(b):
            if a[i] <= b[j]:
                res.append(a[i]); i += 1
            else:
                res.append(b[j]); j += 1
        res.extend(a[i:])
        res.extend(b[j:])
        return res

print(SolutionMerge().getAllElements(build_tree([2, 1, 4]), build_tree([1, 0, 3])))

[0, 1, 1, 2, 3, 4]


**Độ phức tạp cách 2:** Time `O(m + n)`, Space `O(m + n)` (hai list trung gian + kết quả; stack `O(h)`).

### 7.2 Cách 3 — Duyệt inorder song song bằng hai stack

Không tạo `a`, `b`. Mỗi cây có một stack chứa "đường đi về trái". Hàm `push_left(stack, node)` đẩy `node` và toàn bộ nhánh trái của nó.

Vòng lặp chính, khi còn ít nhất một stack khác rỗng:
- Chọn stack có **đỉnh nhỏ hơn** (nếu một stack rỗng thì chọn stack kia).
- `pop` đỉnh đó, ghi vào kết quả, rồi `push_left(stack đó, node.right)`.

In [12]:
def all_elements_two_stacks(root1, root2):
    def push_left(stack, node):
        while node:
            stack.append(node)
            node = node.left
    s1, s2 = [], []
    push_left(s1, root1)
    push_left(s2, root2)
    out = []
    while s1 or s2:
        # chọn s1 nếu s2 rỗng, hoặc cả hai khác rỗng và đỉnh s1 <= đỉnh s2
        if s1 and (not s2 or s1[-1].val <= s2[-1].val):
            stack = s1
        else:
            stack = s2
        node = stack.pop()
        out.append(node.val)
        push_left(stack, node.right)
    return out

for (v1, v2), want in examples:
    print(all_elements_two_stacks(build_tree(v1), build_tree(v2)), "| mong đợi", want)

[0, 1, 1, 2, 3, 4] | mong đợi [0, 1, 1, 2, 3, 4]
[1, 1, 8, 8] | mong đợi [1, 1, 8, 8]


Code nộp LeetCode (cách 3):

In [13]:
class SolutionTwoStacks:
    def getAllElements(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> List[int]:
        def push_left(stack, node):
            while node:
                stack.append(node)
                node = node.left
        s1, s2 = [], []
        push_left(s1, root1)
        push_left(s2, root2)
        res = []
        while s1 or s2:
            stack = s1 if s1 and (not s2 or s1[-1].val <= s2[-1].val) else s2
            node = stack.pop()
            res.append(node.val)
            push_left(stack, node.right)
        return res

print(SolutionTwoStacks().getAllElements(build_tree([1, None, 8]), build_tree([8, 1])))

[1, 1, 8, 8]


**Độ phức tạp cách 3:** Time `O(m + n)` (mỗi nút được đẩy và lấy ra đúng một lần), Space phụ `O(h1 + h2)` cho hai stack (không kể danh sách kết quả bắt buộc phải trả về).

### 7.3 Cách 4 — Generator inorder + `heapq.merge`

Cách "Pythonic": hai generator inorder lười, để `heapq.merge` trộn. Ngắn gọn, vẫn `O(m + n)`, bộ nhớ phụ `O(h1 + h2)` giống cách 3.

In [14]:
def all_elements_heapq(root1, root2):
    return list(heapq.merge(inorder_gen(root1), inorder_gen(root2)))

for (v1, v2), want in examples:
    print(all_elements_heapq(build_tree(v1), build_tree(v2)), "| mong đợi", want)

[0, 1, 1, 2, 3, 4] | mong đợi [0, 1, 1, 2, 3, 4]
[1, 1, 8, 8] | mong đợi [1, 1, 8, 8]


Code nộp LeetCode (cách 4):

In [15]:
class SolutionHeapq:
    def getAllElements(self, root1: Optional[TreeNode], root2: Optional[TreeNode]) -> List[int]:
        def gen(root):
            stack, cur = [], root
            while cur or stack:
                while cur:
                    stack.append(cur)
                    cur = cur.left
                cur = stack.pop()
                yield cur.val
                cur = cur.right
        return list(heapq.merge(gen(root1), gen(root2)))

print(SolutionHeapq().getAllElements(build_tree([2, 1, 4]), None))

[1, 2, 4]


**Độ phức tạp cách 4:** Time `O(m + n)` (với 2 nguồn, `heapq.merge` mỗi bước `O(log 2) = O(1)`), Space phụ `O(h1 + h2)`.

In [16]:
rng = random.Random(5)
r1 = bst_from(rng.sample(range(-10**5, 10**5), 5000))
r2 = bst_from(rng.sample(range(-10**5, 10**5), 5000))
for name, f in [("sort", all_elements_sort), ("merge", all_elements_merge),
                ("two_stacks", all_elements_two_stacks), ("heapq", all_elements_heapq)]:
    t0 = time.perf_counter()
    res = f(r1, r2)
    print(f"{name:10s} -> {len(res)} phần tử ({time.perf_counter() - t0:.4f}s)")

sort       -> 10000 phần tử (0.0020s)
merge      -> 10000 phần tử (0.0025s)
two_stacks -> 10000 phần tử (0.0020s)
heapq      -> 10000 phần tử (0.0030s)


## 8. TRACE từng bước

Trace cách 3 (hai stack) trên Ví dụ 1. Ký hiệu stack viết từ đáy → đỉnh.

| Bước | s1 | s2 | Đỉnh so sánh | Lấy | push_left(node.right) | out |
|---|---|---|---|---|---|---|
| khởi tạo | [2,1] | [1,0] | | | | [] |
| 1 | [2,1] | [1,0] | 1 vs 0 | 0 (s2) | right(0)=None | [0] |
| 2 | [2,1] | [1] | 1 vs 1 | 1 (s1) | right(1)=None | [0,1] |
| 3 | [2] | [1] | 2 vs 1 | 1 (s2) | right(1)=3 → s2=[3] | [0,1,1] |
| 4 | [2] | [3] | 2 vs 3 | 2 (s1) | right(2)=4 → s1=[4] | [0,1,1,2] |
| 5 | [4] | [3] | 4 vs 3 | 3 (s2) | None | [0,1,1,2,3] |
| 6 | [4] | [] | s2 rỗng | 4 (s1) | None | [0,1,1,2,3,4] |

In [17]:
def trace_two_stacks(root1, root2):
    def push_left(stack, node):
        while node:
            stack.append(node)
            node = node.left
    s1, s2 = [], []
    push_left(s1, root1)
    push_left(s2, root2)
    out = []
    step = 0
    vals = lambda s: [n.val for n in s]
    print(f"khởi tạo: s1={vals(s1)} s2={vals(s2)}")
    while s1 or s2:
        step += 1
        use1 = bool(s1) and (not s2 or s1[-1].val <= s2[-1].val)
        stack, name = (s1, "s1") if use1 else (s2, "s2")
        node = stack.pop()
        out.append(node.val)
        push_left(stack, node.right)
        print(f"bước {step}: lấy {node.val} từ {name} -> s1={vals(s1)} s2={vals(s2)} out={out}")
    return out

trace_two_stacks(build_tree([2, 1, 4]), build_tree([1, 0, 3]))

khởi tạo: s1=[2, 1] s2=[1, 0]
bước 1: lấy 0 từ s2 -> s1=[2, 1] s2=[1] out=[0]
bước 2: lấy 1 từ s1 -> s1=[2] s2=[1] out=[0, 1]
bước 3: lấy 1 từ s2 -> s1=[2] s2=[3] out=[0, 1, 1]
bước 4: lấy 2 từ s1 -> s1=[4] s2=[3] out=[0, 1, 1, 2]
bước 5: lấy 3 từ s2 -> s1=[4] s2=[] out=[0, 1, 1, 2, 3]
bước 6: lấy 4 từ s1 -> s1=[] s2=[] out=[0, 1, 1, 2, 3, 4]


[0, 1, 1, 2, 3, 4]

In [18]:
trace_two_stacks(build_tree([1, None, 8]), build_tree([8, 1]))

khởi tạo: s1=[1] s2=[8, 1]
bước 1: lấy 1 từ s1 -> s1=[8] s2=[8, 1] out=[1]
bước 2: lấy 1 từ s2 -> s1=[8] s2=[8] out=[1, 1]
bước 3: lấy 8 từ s1 -> s1=[] s2=[8] out=[1, 1, 8]
bước 4: lấy 8 từ s2 -> s1=[] s2=[] out=[1, 1, 8, 8]


[1, 1, 8, 8]

## 9. Code nộp LeetCode

Bản chính thức: **duyệt inorder song song bằng hai stack** — `O(m + n)`, không đệ quy, bộ nhớ phụ chỉ `O(h1 + h2)`. (Trên LeetCode lớp `TreeNode` có sẵn.)

In [19]:
class Solution:
    def getAllElements(self, root1: TreeNode | None, root2: TreeNode | None) -> list[int]:
        def push_left(stack, node):
            while node:                       # đẩy node và cả nhánh trái
                stack.append(node)
                node = node.left

        s1, s2 = [], []
        push_left(s1, root1)
        push_left(s2, root2)
        res = []
        while s1 or s2:
            # đỉnh stack = phần tử nhỏ nhất chưa lấy của mỗi cây
            if s1 and (not s2 or s1[-1].val <= s2[-1].val):
                stack = s1
            else:
                stack = s2
            node = stack.pop()
            res.append(node.val)
            push_left(stack, node.right)      # tiến tới phần tử kế tiếp của cây đó
        return res

sol = Solution()
for (v1, v2), want in examples:
    got = sol.getAllElements(build_tree(v1), build_tree(v2))
    print(got, "✔" if got == want else "✘")

[0, 1, 1, 2, 3, 4] ✔
[1, 1, 8, 8] ✔


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (inorder BST tăng dần):** quy nạp theo chiều cao — inorder(trái) gồm các giá trị nhỏ hơn gốc và tăng (giả thiết quy nạp), rồi đến gốc, rồi inorder(phải) gồm các giá trị lớn hơn gốc và tăng ⇒ cả dãy tăng.

**Bổ đề 2 (đỉnh stack là phần tử nhỏ nhất chưa lấy):** stack chứa một đường đi "về trái"; các nút chưa thăm gồm: nút trong stack và **cây con phải** của chúng. Đỉnh stack nhỏ hơn mọi nút khác trong stack (nó nằm trong cây con trái của chúng) và nhỏ hơn cây con phải của chính nó. Sau `pop` và `push_left(node.right)`, cấu trúc này được giữ nguyên ⇒ invariant đúng mãi.

**Trộn đúng:** mỗi bước lấy `min(đỉnh s1, đỉnh s2)` = phần tử nhỏ nhất trong toàn bộ phần chưa lấy của **cả hai** cây ⇒ `res` luôn là tiền tố đã sắp xếp của kết quả cuối. Mỗi nút được đẩy đúng một lần và lấy ra đúng một lần ⇒ đủ `m + n` phần tử, giữ cả giá trị trùng.

## 11. Phân tích Time / Space

Gọi `m`, `n` = số nút hai cây (mỗi cây ≤ 5000), `h1`, `h2` = chiều cao.

| Cách | Time | Space phụ (không kể kết quả) | Ghi chú |
|---|---|---|---|
| 1. Gom + sort | `O(N log N)` | `O(N)` | không dùng tính chất BST |
| 2. Inorder + merge | `O(N)` | `O(N)` (hai list trung gian) | dễ hiểu nhất |
| 3. Hai stack song song | `O(N)` | `O(h1 + h2)` | bản nộp chính |
| 4. Generator + heapq.merge | `O(N)` | `O(h1 + h2)` | ngắn gọn, Pythonic |

Với `N = m + n`. Cây cân bằng: `h ≈ log n`; cây lệch: `h = n` (khi đó stack `O(n)`, vẫn không tệ hơn cách 2). Kết quả trả về luôn chiếm `O(N)`.

## 12. Bộ test

Ví dụ đề, edge case (một hoặc cả hai cây rỗng, một nút, giá trị âm, nhiều giá trị trùng giữa hai cây, cây lệch 5000 nút), và đối chiếu mọi cách với `sorted(...)` trên BST ngẫu nhiên (seed cố định).

In [20]:
ALL = [all_elements_sort, all_elements_merge, all_elements_two_stacks, all_elements_heapq,
       Solution().getAllElements, SolutionMerge().getAllElements,
       SolutionTwoStacks().getAllElements, SolutionHeapq().getAllElements]
tests = [
    (([2, 1, 4], [1, 0, 3]), [0, 1, 1, 2, 3, 4]),
    (([1, None, 8], [8, 1]), [1, 1, 8, 8]),
    (([], []), []),                                 # cả hai rỗng
    (([], [5, 1, 7]), [1, 5, 7]),                   # một cây rỗng
    (([0, -10, 10], []), [-10, 0, 10]),
    (([3], [3]), [3, 3]),                           # trùng
    (([-100000], [100000]), [-100000, 100000]),     # biên giá trị
    (([5, 3, 8, 1, 4, None, 9], [6, 2, 7]), [1, 2, 3, 4, 5, 6, 7, 8, 9]),
]
for (v1, v2), want in tests:
    for f in ALL:
        got = f(build_tree(v1), build_tree(v2))
        assert got == want, (f, v1, v2, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 8 test cố định qua cho 8 hàm


In [21]:
rng = random.Random(1305)
for _ in range(400):
    a = rng.sample(range(-50, 51), rng.randint(0, 30))
    b = rng.sample(range(-50, 51), rng.randint(0, 30))
    r1, r2 = bst_from(a), bst_from(b)
    want = sorted(a + b)
    for f in ALL:
        assert f(r1, r2) == want
print("Đối chiếu 400 cặp BST ngẫu nhiên: khớp")

# cây lệch 5000 nút — dựng trực tiếp thành "chuỗi" (giống chèn tăng/giảm dần vào BST)
def chain(values, side):
    root = cur = None
    for x in values:
        node = TreeNode(x)
        if root is None:
            root = cur = node
        else:
            setattr(cur, side, node)
            cur = node
    return root

skew1 = chain(range(0, 10000, 2), "right")     # lệch phải: 0 -> 2 -> 4 ...
skew2 = chain(range(9999, 0, -2), "left")      # lệch trái: 9999 -> 9997 ...
t0 = time.perf_counter()
res = Solution().getAllElements(skew1, skew2)
assert res == list(range(10000))
print("Cây lệch 5000 + 5000 nút: OK", f"({time.perf_counter() - t0:.3f}s)")

Đối chiếu 400 cặp BST ngẫu nhiên: khớp
Cây lệch 5000 + 5000 nút: OK (0.002s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng preorder (hoặc BFS) rồi trộn như thể đã sắp xếp

In [22]:
def wrong_preorder_merge(root1, root2):
    a, b = preorder(root1, []), preorder(root2, [])   # SAI: preorder không tăng dần
    return merge(a, b)

r1, r2 = build_tree([2, 1, 4]), build_tree([1, 0, 3])
print("sai :", wrong_preorder_merge(r1, r2))
print("đúng:", Solution().getAllElements(r1, r2))

sai : [1, 0, 2, 1, 3, 4]
đúng: [0, 1, 1, 2, 3, 4]


**Vì sao sai:** preorder `[2,1,4]` không tăng; phép trộn hai con trỏ chỉ đúng khi **cả hai dãy đã tăng**. **Sửa:** dùng **inorder**.

### Lỗi 2 — Quên chép phần còn dư sau vòng trộn

In [23]:
def wrong_no_tail(root1, root2):
    a, b = inorder_iter(root1), inorder_iter(root2)
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    return out                                     # SAI: thiếu phần dư

r1, r2 = build_tree([2, 1, 4]), build_tree([1, 0, 3])
print("sai :", wrong_no_tail(r1, r2))
print("sai (cây 2 rỗng):", wrong_no_tail(r1, None))
print("đúng:", Solution().getAllElements(r1, r2))

sai : [0, 1, 1, 2, 3]
sai (cây 2 rỗng): []
đúng: [0, 1, 1, 2, 3, 4]


**Vì sao sai:** vòng `while` dừng khi **một** dãy hết; phần tử lớn nhất (`4`) còn nằm trong dãy kia. Nếu một cây rỗng thì kết quả rỗng luôn. **Sửa:** `out.extend(a[i:]); out.extend(b[j:])`.

### Lỗi 3 — Hai stack: chỉ đẩy `node.right` thay vì cả nhánh trái của nó

In [24]:
def wrong_push_right_only(root1, root2):
    def push_left(stack, node):
        while node:
            stack.append(node); node = node.left
    s1, s2 = [], []
    push_left(s1, root1); push_left(s2, root2)
    out = []
    while s1 or s2:
        stack = s1 if s1 and (not s2 or s1[-1].val <= s2[-1].val) else s2
        node = stack.pop()
        out.append(node.val)
        if node.right:
            stack.append(node.right)             # SAI: phải push_left(stack, node.right)
    return out

r1, r2 = bst_from([5, 3, 8, 7, 6]), bst_from([1])
print("sai :", wrong_push_right_only(r1, r2))
print("đúng:", Solution().getAllElements(r1, r2))

sai : [1, 3, 5, 8]
đúng: [1, 3, 5, 6, 7, 8]


**Vì sao sai:** sau khi lấy `5`, ta đẩy `8` nhưng `8` có cây con trái `7 → 6` nhỏ hơn; đỉnh stack không còn là phần tử nhỏ nhất ⇒ thứ tự sai (và nút `6`, `7` có thể bị bỏ sót vì không bao giờ được đẩy). **Sửa:** luôn `push_left(stack, node.right)`.

### Lỗi 4 — Không xử lý cây rỗng (`root = None`)

In [25]:
def wrong_empty(root1, root2):
    out = []
    for root in (root1, root2):
        stack = [root]                          # SAI: đẩy cả None vào
        while stack:
            node = stack.pop()
            out.append(node.val)                # None.val -> AttributeError
            if node.left: stack.append(node.left)
            if node.right: stack.append(node.right)
    return sorted(out)

try:
    print(wrong_empty(build_tree([2, 1, 4]), None))
except AttributeError as e:
    print("Bắt được AttributeError:", e)
print("đúng:", Solution().getAllElements(build_tree([2, 1, 4]), None))

Bắt được AttributeError: 'NoneType' object has no attribute 'val'
đúng: [1, 2, 4]


**Vì sao sai:** đề cho phép cây có **0 nút**. **Sửa:** chỉ đẩy nút khác `None` (`deque([root] if root else [])`), hoặc dùng mẫu `while cur or stack` như inorder lặp.

### Lỗi 5 — Inorder đệ quy với cây lệch sâu

Cây lệch 5000 nút + giới hạn đệ quy mặc định ~1000 ⇒ `RecursionError`.

In [26]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    inorder(skew1, [])
    print("không lỗi?")
except RecursionError as e:
    print("Bắt được RecursionError:", str(e)[:50])
finally:
    sys.setrecursionlimit(old)
print("bản lặp vẫn chạy:", len(inorder_iter(skew1)), "phần tử")

Bắt được RecursionError: maximum recursion depth exceeded
bản lặp vẫn chạy: 5000 phần tử


**Sửa:** dùng inorder lặp bằng stack (như bản nộp) hoặc nâng `sys.setrecursionlimit`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **BST** + **"sắp xếp / thứ tự / k-th nhỏ nhất"** ⇒ nghĩ ngay **inorder**.
- **Hai (hoặc k) nguồn đã sắp xếp** ⇒ **merge hai con trỏ** (hoặc heap cho k nguồn).
- Muốn tiết kiệm bộ nhớ / dừng sớm ⇒ **inorder lặp lười** (stack hoặc generator) — mẫu "BST Iterator" (LeetCode 173).
- `heapq.merge` là công cụ có sẵn để trộn nhiều iterator đã sắp xếp.
- Luôn kiểm tra **cây rỗng** khi đề nói "số nút trong `[0, …]`".

## 15. Biến thể của bài

1. **LeetCode 173 — Binary Search Tree Iterator**: chính là "một nửa" của cách 3 (`next()` = pop + push_left).
2. **LeetCode 21 / 88 — Merge Two Sorted Lists / Merge Sorted Array**: bước trộn hai con trỏ.
3. **LeetCode 23 — Merge k Sorted Lists**: tổng quát lên k nguồn, dùng heap.
4. **LeetCode 230 — Kth Smallest Element in a BST**: inorder dừng ở phần tử thứ k.
5. **LeetCode 1214 — Two Sum BSTs**: duyệt một cây tăng, một cây giảm bằng hai stack.
6. **k-th nhỏ nhất trong hợp hai BST** mà không dựng toàn bộ danh sách (bài tập 5).

In [27]:
# Biến thể 3 (k cây): trộn k BST bằng heapq.merge các generator
trees = [bst_from([5, 1, 9]), bst_from([2, 8]), None, bst_from([7, 3, 4])]
print(list(heapq.merge(*(inorder_gen(t) for t in trees))))

[1, 2, 3, 4, 5, 7, 8, 9]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Gom + sort | Duyệt bất kỳ, `sorted` | `O(N log N)` | `O(N)` | Nhanh gọn, không cần BST |
| 2. Inorder + merge | Hai dãy tăng, trộn hai con trỏ | `O(N)` | `O(N)` | Dễ giải thích |
| 3. Hai stack song song | Đỉnh stack = min chưa lấy | `O(N)` | `O(h1 + h2)` | Bản nộp tối ưu, dừng sớm được |
| 4. Generator + heapq.merge | Lười + thư viện chuẩn | `O(N)` | `O(h1 + h2)` | Code ngắn nhất, mở rộng k cây |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().getAllElements(build_tree([]), build_tree([5, 1, 7]))` và `Solution().getAllElements(build_tree([3, 1]), build_tree([2]))`?

**Bài 2 (tính tay):** Ghi lại thứ tự lấy phần tử (từ s1 hay s2) của cách 3 khi `root1 = [4, 2, 6]`, `root2 = [3, 1, 5]`.

**Bài 3 (điền TODO):** Hoàn thành generator `my_inorder_gen(root)` (lặp bằng stack).

**Bài 4 (tự viết lại):** Viết lại hàm `my_merge(a, b)` trộn hai list tăng dần.

**Bài 5 (mini challenge):** Viết `kth_smallest_two(root1, root2, k)` trả phần tử nhỏ thứ `k` (1-indexed) trong hợp hai BST, **dừng ngay** khi đủ `k` phần tử (không dựng toàn bộ danh sách).

In [28]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def my_inorder_gen_todo(root):
    stack, cur = [], root
    while cur or stack:
        # TODO: đi hết về trái, pop, yield, sang phải
        pass

def my_merge_todo(a, b):
    # TODO: hai con trỏ + phần dư
    pass

def kth_smallest_two_todo(root1, root2, k):
    # TODO: hai stack song song, dừng ở phần tử thứ k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Cây rỗng đóng góp `[]`. (2) Inorder `[5,1,7]` là `[1,5,7]`. (3) `[3,1]` cho `[1,3]`, trộn với `[2]`.

**Bài 2:** (1) Khởi tạo `s1 = [4,2]`, `s2 = [3,1]`. (2) So đỉnh: `2` vs `1`. (3) Thứ tự lấy: 1(s2), 2(s1), 3(s2), 4(s1), 5(s2), 6(s1).

**Bài 3:** (1) Vòng trong `while cur`: đẩy và đi trái. (2) `cur = stack.pop(); yield cur.val`. (3) `cur = cur.right`.

**Bài 4:** (1) `while i < len(a) and j < len(b)`. (2) Lấy cái nhỏ hơn (`<=` để ổn định). (3) Đừng quên `extend` phần dư.

**Bài 5:** (1) Dùng y nguyên vòng lặp của `Solution`. (2) Đếm số phần tử đã lấy. (3) Khi đếm tới `k` thì trả giá trị vừa lấy; nếu `k` lớn hơn tổng số nút, trả `None`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
assert Solution().getAllElements(build_tree([]), build_tree([5, 1, 7])) == [1, 5, 7]
assert Solution().getAllElements(build_tree([3, 1]), build_tree([2])) == [1, 2, 3]
# Bài 2
order = trace_two_stacks(build_tree([4, 2, 6]), build_tree([3, 1, 5]))
assert order == [1, 2, 3, 4, 5, 6]

# Bài 3
def my_inorder_gen(root):
    stack, cur = [], root
    while cur or stack:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        yield cur.val
        cur = cur.right

assert list(my_inorder_gen(bst_from([5, 3, 8, 1, 4, 9]))) == [1, 3, 4, 5, 8, 9]
assert list(my_inorder_gen(None)) == []
print("Bài 1–3: OK")

khởi tạo: s1=[4, 2] s2=[3, 1]
bước 1: lấy 1 từ s2 -> s1=[4, 2] s2=[3] out=[1]
bước 2: lấy 2 từ s1 -> s1=[4] s2=[3] out=[1, 2]
bước 3: lấy 3 từ s2 -> s1=[4] s2=[5] out=[1, 2, 3]
bước 4: lấy 4 từ s1 -> s1=[6] s2=[5] out=[1, 2, 3, 4]
bước 5: lấy 5 từ s2 -> s1=[6] s2=[] out=[1, 2, 3, 4, 5]
bước 6: lấy 6 từ s1 -> s1=[] s2=[] out=[1, 2, 3, 4, 5, 6]
Bài 1–3: OK


In [30]:
# Bài 4
def my_merge(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:]); out.extend(b[j:])
    return out

# Bài 5
def kth_smallest_two(root1, root2, k):
    def push_left(stack, node):
        while node:
            stack.append(node); node = node.left
    s1, s2 = [], []
    push_left(s1, root1); push_left(s2, root2)
    count = 0
    while s1 or s2:
        stack = s1 if s1 and (not s2 or s1[-1].val <= s2[-1].val) else s2
        node = stack.pop()
        count += 1
        if count == k:
            return node.val
        push_left(stack, node.right)
    return None

rng = random.Random(55)
for _ in range(300):
    a = sorted(rng.sample(range(-30, 31), rng.randint(0, 12)))
    b = sorted(rng.sample(range(-30, 31), rng.randint(0, 12)))
    assert my_merge(a, b) == sorted(a + b)
    r1, r2 = bst_from(rng.sample(a, len(a))), bst_from(rng.sample(b, len(b)))
    allv = sorted(a + b)
    for k in range(1, len(allv) + 2):
        want = allv[k - 1] if k <= len(allv) else None
        assert kth_smallest_two(r1, r2, k) == want
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template inorder lặp (BST iterator):**

```python
def push_left(stack, node):
    while node:
        stack.append(node)
        node = node.left

stack = []; push_left(stack, root)
while stack:
    node = stack.pop()          # phần tử nhỏ nhất chưa thăm
    visit(node.val)
    push_left(stack, node.right)
```

**Template trộn hai nguồn đã sắp xếp:**

```python
while s1 or s2:
    stack = s1 if s1 and (not s2 or top(s1) <= top(s2)) else s2
    ...  # lấy từ stack đã chọn, tiến nguồn đó
```

**Checklist trước khi submit:**
- [ ] Dùng **inorder** (không phải pre/post/BFS) nếu muốn dãy tăng.
- [ ] Trộn xong nhớ **phần dư** / vòng lặp chạy tới khi **cả hai** nguồn rỗng.
- [ ] `push_left(node.right)` chứ không chỉ đẩy `node.right`.
- [ ] Xử lý **cây rỗng**; giữ **giá trị trùng**.
- [ ] Không đệ quy sâu (cây lệch 5000 nút).
- [ ] Đúng chữ ký: `getAllElements(self, root1: TreeNode | None, root2: TreeNode | None) -> list[int]`.